[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S18_ml_clasificacion.ipynb)

# Sesión 18 · Clasificación

**Módulo 5: Machine Learning** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Dividir datos de clasificación manteniendo la proporción de clases (`stratify`).
2. Entrenar una regresión logística, un árbol de decisión y un KNN, y comparar sus aciertos.
3. Obtener probabilidades con `predict_proba` e interpretar coeficientes e importancias.
4. Elegir el umbral de decisión según lo que necesita el negocio.

## 📋 Qué debes saber antes
Sesión 17: `X` e `y`, `train_test_split`, `fit` y `predict`, y el baseline.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Hoy medimos los modelos con la **accuracy** (proporción de aciertos). En la sesión 19 verás por qué no alcanza.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Datos de práctica: ¿qué clientes abandonarán el banco? ----------
_n = 1200
_ant = rng.integers(1, 121, _n)
_prod = rng.integers(1, 6, _n)
_rec = rng.poisson(0.6, _n)
_saldo = np.round(np.clip(rng.lognormal(2.2, 0.9, _n), 0.1, 80), 1)
_app = (rng.random(_n) < 0.6).astype(int)
_logit = 1.1 - 0.025 * _ant - 0.45 * (_prod - 1) + 0.9 * _rec - 0.04 * _saldo - 0.9 * _app
clientes = pd.DataFrame({
    "antiguedad_meses": _ant, "n_productos": _prod, "reclamos_12m": _rec, "saldo_miles": _saldo, "usa_app": _app,
    "abandona": (rng.random(_n) < 1 / (1 + np.exp(-_logit))).astype(int),
})
_k = 8
clientes_nuevos = pd.DataFrame({
    "id": [f"N{i + 1:02d}" for i in range(_k)],
    "antiguedad_meses": rng.integers(1, 121, _k), "n_productos": rng.integers(1, 6, _k),
    "reclamos_12m": rng.poisson(1.0, _k), "saldo_miles": np.round(rng.uniform(0.5, 40, _k), 1), "usa_app": rng.integers(0, 2, _k),
})
VARIABLES = ["antiguedad_meses", "n_productos", "reclamos_12m", "saldo_miles", "usa_app"]

_D = copy.deepcopy({"clientes": clientes, "clientes_nuevos": clientes_nuevos})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _vector(r, nombre, largo, binario=False):
    v = r.var(nombre)
    if v is _FALTA:
        return None
    try:
        arr = np.asarray(v, dtype=float).ravel()
    except (TypeError, ValueError):
        arr = None
    if arr is None or len(arr) != largo:
        r.mal(f"`{nombre}` debería tener {largo} valores, uno por fila de prueba.")
        return None
    if binario and not set(np.unique(arr)) <= {0.0, 1.0}:
        r.mal(f"`{nombre}` debería tener solo ceros y unos.")
        return None
    return arr


def _datos():
    """X_test e y_test del alumno, si existen y tienen forma válida."""
    xt, yt = globals().get("X_test"), globals().get("y_test")
    if isinstance(xt, pd.DataFrame) and isinstance(yt, pd.Series) and len(xt) == len(yt):
        return xt, np.asarray(yt, dtype=float)
    return None


def _acierto(pred, real):
    return sum(1 for a, b in zip(pred, real) if a == b) / len(real)


def _modelo(r, nombre, tipo):
    m = r.var(nombre)
    if m is _FALTA:
        return None
    if type(m).__name__ != tipo:
        r.mal(f"`{nombre}` debería ser un `{tipo}` y es {type(m).__name__}.")
        return None
    if not hasattr(m, "classes_"):
        r.mal(f"`{nombre}` todavía no está entrenado: llama a `fit`.")
        return None
    return m


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    c = _D["clientes"]
    _df(r, "X", VARIABLES, c[VARIABLES].values.tolist(), "las cinco variables, en el orden de `VARIABLES`")
    _ser(r, "y", c["abandona"].tolist(), "la columna abandona")
    tasa = statistics.fmean(c["abandona"].tolist())
    _esc(r, "tasa_abandono", round(tasa, 3), "la proporción de clientes que abandonan, con 3 decimales", tol=0.00051)
    xs, xt, ys, yt = (globals().get(n) for n in ["X_train", "X_test", "y_train", "y_test"])
    if not all(isinstance(v, (pd.DataFrame, pd.Series)) for v in (xs, xt, ys, yt)):
        r.mal("Faltan `X_train`, `X_test`, `y_train` o `y_test`.")
    elif len(xt) != 300 or len(xs) != 900:
        r.mal(f"`X_test` tiene {len(xt)} filas y `X_train`, {len(xs)}: con test_size=0.25 se esperaban 300 y 900.")
    elif list(xs.index) != list(ys.index) or list(xt.index) != list(yt.index) or set(xs.index) & set(xt.index):
        r.mal("Entrenamiento y prueba deberían repartirse las filas sin repetir ninguna, con cada X alineada con su y.")
    elif abs(statistics.fmean(ys.tolist()) - tasa) > 0.005 or abs(statistics.fmean(yt.tolist()) - tasa) > 0.005:
        r.mal("La proporción de abandonos es distinta en entrenamiento y en prueba: usa `stratify=y`.")
    else:
        r.ok("La partición es estratificada: los dos conjuntos tienen la misma proporción de abandonos.")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_n_test": "9959e293f6e1d9a911e2a3a033b419b0500855cf993caa4bce98292bf06bec09",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    d = _datos()
    m = _modelo(r, "modelo_log", "LogisticRegression")
    if d is None:
        r.mal("Primero resuelve el ejercicio 1.")
    elif m is not None:
        xt, yt = d
        prob = _vector(r, "prob_log", len(yt))
        pred = _vector(r, "pred_log", len(yt), binario=True)
        if prob is not None:
            if not ((prob >= 0) & (prob <= 1)).all():
                r.mal("`prob_log` debería tener probabilidades entre 0 y 1: la columna de la clase 1 de `predict_proba`.")
            elif not _cerca_lista(prob, m.predict_proba(xt)[:, 1], 1e-9):
                r.mal("`prob_log` debería ser la probabilidad de **abandonar** (columna 1 de `predict_proba`) para `X_test`.")
            else:
                r.ok("`prob_log` tiene la probabilidad de abandonar de cada cliente de prueba.")
        if pred is not None and prob is not None:
            r.ok("`pred_log` coincide con usar el umbral 0.5.") if all((p > 0.5) == bool(c) for p, c in zip(prob, pred)) else \
                r.mal("`pred_log` debería ser lo que devuelve `predict` para `X_test`.")
        if pred is not None:
            _esc(r, "acc_log", _acierto(pred, yt), "la proporción de aciertos de `pred_log` contra `y_test`", tol=1e-9)
        e = r.var("efectos")
        if e is not _FALTA:
            if not isinstance(e, pd.Series) or [str(i) for i in e.index] != VARIABLES:
                r.mal("`efectos` debería ser una Series con un coeficiente por variable, con los nombres como índice.")
            elif not _cerca_lista(e.tolist(), m.coef_[0], 1e-9):
                r.mal("Los valores de `efectos` deberían ser `modelo_log.coef_[0]`.")
            elif not (e["reclamos_12m"] > 0 and e["usa_app"] < 0):
                r.mal("Los signos de `efectos` no son los esperados: ¿entrenaste con `X_train` e `y_train`?")
            else:
                r.ok("`efectos` es correcto: los reclamos suben el riesgo y usar la app lo baja.")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_suma_proba": "b399e5afcd7822a4082968fd88be832ae2163fda3da62ed3f6d9402bfba8dd1c",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3")
    d = _datos()
    m = _modelo(r, "arbol", "DecisionTreeClassifier")
    if d is None:
        r.mal("Primero resuelve el ejercicio 1.")
    elif m is not None:
        xt, yt = d
        if m.max_depth != 3:
            r.mal("`arbol` debería tener `max_depth=3`.")
        pred = _vector(r, "pred_arbol", len(yt), binario=True)
        if pred is not None:
            if not _cerca_lista(pred, m.predict(xt), 0):
                r.mal("`pred_arbol` debería ser lo que devuelve `arbol.predict(X_test)`.")
            else:
                r.ok("`pred_arbol` es correcto.")
                _esc(r, "acc_arbol", _acierto(pred, yt), "la proporción de aciertos del árbol en prueba", tol=1e-9)
        imp = r.var("importancias")
        if imp is not _FALTA:
            ref = dict(zip(VARIABLES, m.feature_importances_))
            if not isinstance(imp, pd.Series) or sorted(map(str, imp.index)) != sorted(VARIABLES):
                r.mal("`importancias` debería ser una Series con una importancia por variable, con los nombres como índice.")
            elif not all(abs(float(imp[k]) - ref[k]) < 1e-9 for k in VARIABLES):
                r.mal("Los valores de `importancias` deberían salir de `arbol.feature_importances_`.")
            elif any(a < b for a, b in zip(imp.tolist(), imp.tolist()[1:])):
                r.mal("`importancias` debería estar ordenada de mayor a menor.")
            else:
                r.ok("`importancias` es correcta y está ordenada.")
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    d = _datos()
    xs = globals().get("X_train")
    if d is None or not isinstance(xs, pd.DataFrame):
        r.mal("Primero resuelve el ejercicio 1.")
    else:
        xt, yt = d
        cols = list(zip(*np.asarray(xs, dtype=float).tolist()))
        medias = [statistics.fmean(c) for c in cols]
        desvs = [statistics.pstdev(c) for c in cols]
        a = r.var("X_train_esc")
        if a is not _FALTA:
            a = np.asarray(a, dtype=float)
            if a.shape != xs.shape or not np.allclose(a.mean(axis=0), 0, atol=1e-9) or not np.allclose(a.std(axis=0), 1, atol=1e-9):
                r.mal("`X_train_esc` debería tener cada columna con media 0 y desviación 1.")
            else:
                r.ok("`X_train_esc` está estandarizado.")
        b = r.var("X_test_esc")
        if b is not _FALTA:
            b = np.asarray(b, dtype=float)
            esperado = [[(x - m) / s for x, m, s in zip(fila, medias, desvs)] for fila in np.asarray(xt, dtype=float).tolist()]
            if b.shape != xt.shape or not np.allclose(b, esperado, atol=1e-9):
                r.mal("`X_test_esc` debería transformarse con las medias y desviaciones de **entrenamiento** (`scaler.transform`, sin volver a ajustar).")
            else:
                r.ok("`X_test_esc` usa la escala aprendida en entrenamiento.")
        m = _modelo(r, "knn", "KNeighborsClassifier")
        if m is not None:
            if m.n_neighbors != 15:
                r.mal("`knn` debería usar 15 vecinos.")
            pred = _vector(r, "pred_knn", len(yt), binario=True)
            if pred is not None and isinstance(b, np.ndarray) and b.shape == xt.shape:
                if not _cerca_lista(pred, m.predict(b), 0):
                    r.mal("`pred_knn` debería ser `knn.predict(X_test_esc)`.")
                else:
                    r.ok("`pred_knn` es correcto.")
                    _esc(r, "acc_knn", _acierto(pred, yt), "la proporción de aciertos de KNN en prueba", tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_k1_train": "b399e5afcd7822a4082968fd88be832ae2163fda3da62ed3f6d9402bfba8dd1c",
        "pred_escalar_con": "0a439258a247f70cba6aac61e7d82cb85b1e5c426c8b5749d2372fb803260d65",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    d = _datos()
    prob = globals().get("prob_log")
    if d is None or prob is None or len(np.ravel(prob)) != len(d[1]):
        r.mal("Primero resuelve los ejercicios 1 y 2.")
    else:
        yt = d[1]
        prob = np.asarray(prob, dtype=float).ravel()
        for u, sufijo in ((0.5, "05"), (0.3, "03")):
            marcados = [p >= u for p in prob]
            _esc(r, f"alertas_{sufijo}", sum(marcados), f"cuántos clientes quedan marcados con umbral {u}")
            _esc(r, f"detectados_{sufijo}", sum(1 for m, y in zip(marcados, yt) if m and y == 1),
                 f"cuántos de los que realmente abandonan quedan marcados con umbral {u}")
        det = sum(1 for p, y in zip(prob, yt) if p >= 0.3 and y == 1)
        _esc(r, "pct_detectados_03", round(det * 100 / sum(yt), 1), "el porcentaje de abandonos reales que detecta el umbral 0.3, con 1 decimal", tol=0.051)
        ax = _grafico(r, "ax_umbral")
        if ax is not None:
            verticales = sorted(round(float(l.get_xdata()[0]), 6) for l in ax.get_lines() if len(set(map(float, l.get_xdata()))) == 1)
            leyenda = ax.get_legend()
            if len(_barras(ax)) < 2:
                r.mal("`ax_umbral` debería tener los histogramas de probabilidad de las dos clases.")
            elif verticales != [0.3, 0.5]:
                r.mal("Marca los dos umbrales, 0.3 y 0.5, con líneas verticales.")
            elif leyenda is None or len(leyenda.get_texts()) < 2:
                r.mal("Agrega una leyenda que diga qué histograma es de cada clase.")
            else:
                r.ok("`ax_umbral` muestra las dos clases y los dos umbrales.")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_umbral_bajo": "584127a8f89648c76387dd2586753e7c3d517f910fadecdffb1df98f8bb3bb91",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    nombres = {"logística": "acc_log", "árbol": "acc_arbol", "knn": "acc_knn"}
    accs = {k: globals().get(v) for k, v in nombres.items()}
    if not all(_es_numero(v) for v in accs.values()):
        r.mal("Primero calcula `acc_log`, `acc_arbol` y `acc_knn`.")
    else:
        r.valor("mejor_modelo", max(accs, key=lambda k: float(accs[k])), None,
                "el nombre (\"logística\", \"árbol\" o \"knn\") del modelo con mayor accuracy en prueba", igual=_texto)
    m = globals().get("modelo_log")
    nuevos = _D["clientes_nuevos"]
    if m is None or not hasattr(m, "classes_"):
        r.mal("Primero entrena `modelo_log`.")
    else:
        esperado = m.predict_proba(nuevos[VARIABLES])[:, 1]
        p = r.var("prob_nuevos")
        if p is not _FALTA:
            r.ok("`prob_nuevos` es correcto.") if _cerca_lista(np.ravel(p), esperado, 1e-9) else \
                r.mal("`prob_nuevos` debería ser la probabilidad de abandonar de cada cliente nuevo según `modelo_log` (solo con las columnas de `VARIABLES`).")
        orden = [nuevos["id"].tolist()[i] for i in np.argsort(-esperado, kind="stable")]
        r.valor("top3_riesgo", orden[:3], list, "la lista de los 3 ids con mayor probabilidad, de mayor a menor")
        prob_t = globals().get("prob_log")
        if prob_t is not None:
            u = float(np.quantile(np.asarray(prob_t, dtype=float), 0.8))
            _esc(r, "umbral_20", u, "el percentil 80 de `prob_log`", tol=1e-9)
            _esc(r, "n_marcados_nuevos", int((esperado >= u).sum()), "cuántos clientes nuevos superan `umbral_20`")
    _sin_cambios_df(r, "clientes_nuevos")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    t = r.var("reglas")
    if t is not _FALTA:
        if not isinstance(t, str) or "|---" not in t or not any(v in t for v in VARIABLES):
            r.mal("`reglas` debería ser el texto que devuelve `export_text(arbol, feature_names=...)`, con los nombres de las variables.")
        else:
            r.ok("`reglas` muestra las reglas del árbol con los nombres de las variables.")
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`clientes`: 1200 clientes de un banco con su antigüedad en meses, cantidad de productos, reclamos del último año, saldo promedio (en miles de soles), si usa la app (1) o no (0) y si **abandonó** el banco al año siguiente (1) o no (0). `VARIABLES` tiene los nombres de las cinco variables y `clientes_nuevos`, ocho clientes actuales sobre los que hay que decidir.

In [ ]:
print(clientes.head(), "\n")
print(clientes["abandona"].value_counts(), "\n")
print(clientes.groupby("abandona")[VARIABLES].mean().round(2))

---
## 1. Clasificación y partición estratificada

### 📘 Concepto
En **clasificación**, `y` es una categoría. Con dos clases (0 y 1) se llama clasificación binaria: ¿abandona o no?, ¿es fraude o no?

Cuando una clase es minoritaria (aquí, pocos clientes abandonan), una división al azar puede dejar proporciones distintas en entrenamiento y en prueba. Con **`stratify=y`**, `train_test_split` mantiene la misma proporción de cada clase en los dos conjuntos:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
```

In [ ]:
from sklearn.model_selection import train_test_split

y_ej = pd.Series([0] * 18 + [1] * 2)
X_ej = pd.DataFrame({"x": range(20)})
_, _, _, prueba_sin = train_test_split(X_ej, y_ej, test_size=0.5, random_state=3)
_, _, _, prueba_con = train_test_split(X_ej, y_ej, test_size=0.5, random_state=3, stratify=y_ej)
print(prueba_sin.mean(), prueba_con.mean(), y_ej.mean())

### ✍️ Tu turno · Ejercicio 1: preparar los datos
**Parte A.**
1. `X`: las columnas de `VARIABLES`; `y`: la columna `abandona`.
2. `tasa_abandono`: la proporción de clientes que abandonan, con 3 decimales.
3. `X_train`, `X_test`, `y_train`, `y_test`: 25 % para prueba, `random_state=42` y **estratificado** por `y`.

**Parte B.** Predice **sin ejecutar**: `pred_n_test` = cuántas filas tendrá `X_test`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`clientes[VARIABLES]` selecciona las cinco columnas de una vez.
</details>

<details><summary>💡 Pista 2</summary>

La proporción de unos es el promedio de `y`. Agrega `stratify=y` a `train_test_split`.
</details>

---
## 2. Regresión logística y probabilidades

### 📘 Concepto
La **regresión logística** combina las variables como la regresión lineal, pero pasa el resultado por una curva en forma de S que lo deja entre 0 y 1: una **probabilidad**.

- `modelo.predict_proba(X)` devuelve una fila por caso y una columna por clase; la columna `[:, 1]` es la probabilidad de la clase 1. Cada fila suma 1.
- `modelo.predict(X)` devuelve la clase: 1 si esa probabilidad supera 0.5.
- `modelo.coef_[0]` tiene un coeficiente por variable: si es **positivo**, la variable **sube** la probabilidad de la clase 1; si es negativo, la baja.

Con `LogisticRegression(max_iter=1000)` le das iteraciones de sobra para que el entrenamiento converja.

In [ ]:
from sklearn.linear_model import LogisticRegression

horas_ej = pd.DataFrame({"horas": [1, 2, 3, 4, 5, 6, 7, 8]})
aprueba_ej = pd.Series([0, 0, 0, 1, 0, 1, 1, 1])
log_ej = LogisticRegression(max_iter=1000).fit(horas_ej, aprueba_ej)
print(log_ej.predict_proba(pd.DataFrame({"horas": [2, 5, 8]})).round(3))
print(log_ej.predict(pd.DataFrame({"horas": [2, 5, 8]})), log_ej.coef_)

### ✍️ Tu turno · Ejercicio 2: el primer clasificador
**Parte A.**
1. `modelo_log`: una regresión logística (`max_iter=1000`) entrenada con los datos de entrenamiento.
2. `prob_log`: la probabilidad de **abandonar** de cada cliente de prueba.
3. `pred_log`: la clase predicha para cada cliente de prueba.
4. `acc_log`: la accuracy en prueba, es decir, la proporción de aciertos (compara `pred_log` con `y_test` y promedia).
5. `efectos`: una Series con los coeficientes y los nombres de las variables como índice.

¿Qué variables aumentan el riesgo de abandono y cuáles lo reducen?

**Parte B.** Predice **sin ejecutar**: `pred_suma_proba` = lo que suma cada fila de `predict_proba`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

`predict_proba` devuelve dos columnas; la segunda (`[:, 1]`) es la de abandonar.
</details>

<details><summary>💡 Pista 2</summary>

`acc_log = (pred_log == y_test).mean()`. Para `efectos`: `pd.Series(modelo_log.coef_[0], index=X.columns)`.
</details>

---
## 3. Árbol de decisión

### 📘 Concepto
Un **árbol de decisión** aprende preguntas del tipo "¿reclamos ≥ 2?" y divide a los clientes rama por rama hasta llegar a una predicción. Es fácil de explicar a alguien que no sabe de modelos.

- `max_depth` limita cuántas preguntas seguidas puede hacer. Sin límite, el árbol puede memorizar el entrenamiento y fallar con datos nuevos.
- `feature_importances_` dice cuánto usó el árbol cada variable para separar las clases; las importancias suman 1.
- `random_state` hace que el resultado sea reproducible.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

arbol_ej = DecisionTreeClassifier(max_depth=2, random_state=0).fit(horas_ej, aprueba_ej)
print(arbol_ej.predict(pd.DataFrame({"horas": [2, 5, 8]})), arbol_ej.get_depth(), arbol_ej.feature_importances_)

### ✍️ Tu turno · Ejercicio 3: un árbol que se puede explicar
1. `arbol`: un árbol de decisión con `max_depth=3` y `random_state=42`, entrenado.
2. `pred_arbol` y `acc_arbol`: sus predicciones y su accuracy en prueba.
3. `importancias`: una Series con la importancia de cada variable, ordenada de mayor a menor.

¿Coinciden las variables más importantes del árbol con las de coeficiente más grande de la logística?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Mismo patrón: crear, `fit` y `predict`.
</details>

<details><summary>💡 Pista 2</summary>

`pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)`.
</details>

---
## 4. KNN: los vecinos más parecidos

### 📘 Concepto
**KNN** (*k vecinos más cercanos*) clasifica a un cliente mirando a los `k` clientes de entrenamiento más parecidos y votando. "Parecido" se mide con distancias, así que las escalas importan: una diferencia de 50 meses de antigüedad pesaría mucho más que tener o no la app. Por eso, antes de KNN se **estandarizan** las variables (media 0 y desviación 1), algo que verás a fondo en la sesión 21:

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)          # aprende medias y desviaciones SOLO de entrenamiento
X_train_esc = scaler.transform(X_train)
X_test_esc = scaler.transform(X_test)           # prueba se transforma con las de entrenamiento
```

Nunca ajustes el escalador con los datos de prueba: sería usar información que el modelo no debería conocer.

Con `k` muy chico, el modelo sigue el ruido: con `k=1`, cada cliente de entrenamiento es su propio vecino más cercano.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

esc_ej = StandardScaler().fit(horas_ej)
knn_ej = KNeighborsClassifier(n_neighbors=3).fit(esc_ej.transform(horas_ej), aprueba_ej)
print(knn_ej.predict(esc_ej.transform(pd.DataFrame({"horas": [2, 5, 8]}))))

### ✍️ Tu turno · Ejercicio 4: KNN con variables estandarizadas
**Parte A.**
1. `scaler`: un `StandardScaler` ajustado con `X_train`; `X_train_esc` y `X_test_esc`: los dos conjuntos transformados.
2. `knn`: un KNN con 15 vecinos, entrenado con `X_train_esc`.
3. `pred_knn` y `acc_knn`: sus predicciones y su accuracy en prueba.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_k1_train` | la accuracy **en entrenamiento** de un KNN con `k=1` (no hay dos clientes idénticos) | número |
| `pred_escalar_con` | ¿con los datos de qué conjunto se ajusta el escalador? | `"train"` o `"test"` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`fit` va solo con `X_train`; `transform` se aplica a los dos conjuntos.
</details>

<details><summary>💡 Pista 2</summary>

`knn = KNeighborsClassifier(n_neighbors=15).fit(X_train_esc, y_train)` y luego `knn.predict(X_test_esc)`.
</details>

---
## 5. El umbral de decisión

### 📘 Concepto
`predict` usa un umbral de 0.5, pero ese número no es sagrado. El umbral se elige según lo que cuesta equivocarse:
- **Umbral bajo** (por ejemplo, 0.3): marca a más clientes. Detecta más abandonos reales, pero también da más falsas alarmas.
- **Umbral alto**: marca a menos clientes. Menos falsas alarmas, pero se escapan más abandonos.

Si llamar a un cliente para retenerlo es barato y perderlo es caro, conviene un umbral bajo. Con las probabilidades en la mano, cambiar el umbral es una comparación: `(prob >= 0.3).astype(int)`.

In [ ]:
prob_ej = np.array([0.1, 0.35, 0.45, 0.62, 0.8])
real_ej = np.array([0, 1, 0, 1, 1])
for u in [0.5, 0.3]:
    marcados = prob_ej >= u
    print(u, "marcados:", marcados.sum(), "abandonos detectados:", (marcados & (real_ej == 1)).sum())

### ✍️ Tu turno · Ejercicio 5: mover el umbral
**Parte A.** Con `prob_log` e `y_test`:
1. `alertas_05` y `alertas_03`: cuántos clientes quedan marcados con umbral 0.5 y con umbral 0.3.
2. `detectados_05` y `detectados_03`: cuántos de los que **realmente abandonan** quedan marcados con cada umbral.
3. `pct_detectados_03`: qué porcentaje de todos los abandonos reales detecta el umbral 0.3, con 1 decimal.
4. `fig_umbral, ax_umbral`: dos histogramas de `prob_log` superpuestos, uno para los clientes que no abandonan (`GRIS`) y otro para los que sí (`AZUL`), con `bins=np.linspace(0, 1, 21)`, `alpha=0.7` y leyenda, más dos líneas verticales en 0.3 y 0.5 (`TINTA_2`).

**Parte B.** Responde en `pred_umbral_bajo` con `"más"` o `"menos"`: ¿bajar el umbral da más o menos falsas alarmas?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Una máscara `prob_log >= 0.3` marca a los clientes; combínala con `y_test == 1` usando `&` para contar los detectados.
</details>

<details><summary>💡 Pista 2</summary>

Para los histogramas: `ax_umbral.hist(prob_log[y_test == 0], ...)` y lo mismo con `== 1`. Como `y_test` es una Series, compáralo como array: `y_test.values == 0`.
</details>

---
## 🏋️ Reto final: ¿a quién llamamos esta semana?
El área de retención puede llamar solo al 20 % de los clientes con más riesgo.
1. `mejor_modelo`: el nombre del modelo con mayor accuracy en prueba: `"logística"`, `"árbol"` o `"knn"`.
2. `prob_nuevos`: la probabilidad de abandonar de cada cliente de `clientes_nuevos` según `modelo_log` (usa solo las columnas de `VARIABLES`).
3. `top3_riesgo`: una lista con los `id` de los 3 clientes nuevos de mayor riesgo, de mayor a menor.
4. `umbral_20`: el umbral que marca al 20 % de clientes de prueba con mayor riesgo, es decir, el percentil 80 de `prob_log` (investiga `np.quantile`).
5. `n_marcados_nuevos`: cuántos clientes nuevos superan o igualan `umbral_20`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Arma un diccionario `{"logística": acc_log, ...}` y usa `max` con `key` para quedarte con el nombre.
</details>

<details><summary>💡 Pista 2</summary>

Para el top 3: agrega `prob_nuevos` como columna a una **copia** de `clientes_nuevos`, ordénala con `sort_values(..., ascending=False)` y toma los tres primeros `id` con `.head(3)` y `.tolist()`.
</details>

---
## 🚀 Nivel pro (opcional): leer el árbol
`from sklearn.tree import export_text` convierte un árbol en reglas legibles. Guarda en `reglas` el texto de `export_text(arbol, feature_names=list(X.columns))` e imprímelo. ¿Cuál es la primera pregunta que hace el árbol? ¿Qué rama tiene más riesgo de abandono? Escribe en una celda de texto una regla que podrías contarle al área de retención.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar para qué sirve `stratify` en `train_test_split`.
- [ ] Entrenar una regresión logística, un árbol y un KNN con la misma API.
- [ ] Obtener probabilidades con `predict_proba` y explicar qué hace `predict` con ellas.
- [ ] Interpretar el signo de un coeficiente y las importancias de un árbol.
- [ ] Explicar por qué KNN necesita variables estandarizadas y por qué el escalador se ajusta solo con entrenamiento.
- [ ] Explicar qué pasa con las alertas y los casos detectados al bajar el umbral.

**Próxima sesión (S19):** evaluación: por qué la accuracy engaña, matriz de confusión, precision, recall, F1, ROC-AUC y validación cruzada.